# Phase 4 — Audio Spectrogram Transformer (AST)

**audio → mel-spectrogram → AST → group-aware CV → recall@FPR**

### Kaggle setup
1. Upload this repo as a **Kaggle Dataset** (zip it, upload via "New Dataset")
2. Create a new **Notebook**, attach the dataset, enable **GPU** (T4 P100)
3. Toggle **Internet: ON** for the first run (downloads pretrained DeiT weights)
4. After first run, Internet can be turned off — weights are cached

Alternatively, if the repo is on GitHub, cell 2 clones it directly.

## 1 — Install dependencies & mount repo

In [ ]:
!pip install -q timm librosa pyyaml scikit-learn

import sys, os
from pathlib import Path

# --- Option A: repo uploaded as a Kaggle Dataset ---
KAGGLE_DATASET = Path("/kaggle/input/sonar-competition")

# --- Option B: clone from GitHub ---
GITHUB_REPO = "https://github.com/YOUR_USER/sonar-competition.git"  # <-- update this
CLONE_DIR = Path("/kaggle/working/sonar-competition")

if KAGGLE_DATASET.exists():
    REPO_ROOT = KAGGLE_DATASET
    print(f"Using Kaggle dataset at {REPO_ROOT}")
elif CLONE_DIR.exists():
    REPO_ROOT = CLONE_DIR
    print(f"Using existing clone at {REPO_ROOT}")
else:
    os.system(f"git clone {GITHUB_REPO} {CLONE_DIR}")
    REPO_ROOT = CLONE_DIR
    print(f"Cloned to {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)
print("Contents:", os.listdir(REPO_ROOT))

## 2 — Verify GPU & imports

In [ ]:
import torch
import numpy as np
import yaml

from sonar_toolkit.training.device import get_device, seed_everything, describe_device
from sonar_toolkit.training.loop import fit, TrainConfig
from sonar_toolkit.data.dataset import WindowDataset
from sonar_toolkit.augment.spec import SpecAugment, AddNoise, Compose
from sonar_toolkit.features import mel_spectrogram
from sonar_toolkit.validation import recall_at_fpr
from sonar_toolkit.models.ast import ASTClassifier
from sklearn.model_selection import GroupKFold

device = get_device()
print("Device:", describe_device(device))
seed_everything(42)

## 3 — Configuration

Edit the config inline below, or load from the YAML file in the repo.

In [ ]:
# Load from repo config, then override Kaggle-specific paths
cfg_path = REPO_ROOT / "experiments" / "04_audio_ast" / "config.yaml"
cfg = yaml.safe_load(cfg_path.read_text())

# Override paths for Kaggle working directory
cfg["paths"]["cache_dir"] = "/kaggle/working/cache/phase4_ast"
cfg["paths"]["ckpt_dir"] = "/kaggle/working/checkpoints/phase4_ast"

# --- Override any config values here ---
# cfg["train"]["epochs"] = 15
# cfg["train"]["batch_size"] = 16
# cfg["model"]["pretrained"] = True
# cfg["use_simulator"] = True              # set False if using real audio
# cfg["data_dir"] = "/kaggle/input/your-audio-dataset"

print(yaml.dump(cfg, default_flow_style=False))

## 4 — Load data & build windows

When `use_simulator: true`, generates synthetic recordings.
When `use_simulator: false`, expects audio in `data_dir/positive/` and `data_dir/negative/`.

In [ ]:
def load_audio_files(data_dir, sr):
    """Load from data_dir/positive/ and data_dir/negative/."""
    import librosa
    data_dir = Path(data_dir)
    paths, labels, groups = [], [], []
    group_id = 0
    for label_name, label_val in [("negative", 0), ("positive", 1)]:
        folder = data_dir / label_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Expected {folder}")
        for fp in sorted(folder.glob("*.wav")):
            paths.append(fp)
            labels.append(label_val)
            groups.append(group_id)
            group_id += 1
    return paths, np.array(labels), np.array(groups)


def generate_synthetic(cfg):
    """Generate synthetic recordings via the toolkit simulator."""
    from sonar_toolkit import simulator
    waveforms, labels, groups = [], [], []
    sr, win = cfg["sr"], int(cfg["window_s"] * cfg["sr"])
    hop = int(cfg["hop_s"] * cfg["sr"])
    for rec_id in range(cfg["n_recordings"]):
        y, label = simulator.generate(scenario=cfg["scenario"], sr=sr)
        for i in range(0, max(1, len(y) - win + 1), hop):
            waveforms.append(y[i:i + win])
            labels.append(int(label))
            groups.append(rec_id)
    return waveforms, np.asarray(labels), np.asarray(groups)


def window_audio_files(paths, labels, groups, cfg):
    """Window real audio files into fixed-length segments."""
    import librosa
    sr, win = cfg["sr"], int(cfg["window_s"] * cfg["sr"])
    hop = int(cfg["hop_s"] * cfg["sr"])
    waveforms, win_labels, win_groups = [], [], []
    for fp, lab, grp in zip(paths, labels, groups):
        y, _ = librosa.load(str(fp), sr=sr, mono=True)
        if len(y) < win:
            y = np.pad(y, (0, win - len(y)))
        for i in range(0, max(1, len(y) - win + 1), hop):
            waveforms.append(y[i:i + win].astype(np.float32))
            win_labels.append(lab)
            win_groups.append(grp)
    return waveforms, np.asarray(win_labels), np.asarray(win_groups)


# --- Load ---
if cfg.get("use_simulator", False):
    waveforms, labels, groups = generate_synthetic(cfg)
else:
    paths, labels, groups = load_audio_files(cfg["data_dir"], cfg["sr"])
    waveforms, labels, groups = window_audio_files(paths, labels, groups, cfg)

print(f"Windows: {len(labels)}, positives: {int(labels.sum())}/{len(labels)}")

## 5 — Feature function & augmentation

In [ ]:
f = cfg["feature"]
sr = cfg["sr"]

def feature_fn(wav):
    return mel_spectrogram(wav, sr=sr, n_mels=f["n_mels"], n_fft=f["n_fft"], hop=f["hop_length"])

aug = Compose([
    SpecAugment(freq_mask=16, time_mask=24, n_freq=2, n_time=2),
    AddNoise(snr_db_range=(5, 25)),
])

# Quick sanity check
sample = feature_fn(waveforms[0])
print(f"Mel shape: {sample.shape}  (n_mels={f['n_mels']}, time_frames={sample.shape[1]})")

## 6 — Group-aware cross-validation training loop

In [ ]:
from pathlib import Path

cache_dir = cfg["paths"]["cache_dir"]
ckpt_dir = Path(cfg["paths"]["ckpt_dir"])
ckpt_dir.mkdir(parents=True, exist_ok=True)

target_fpr = cfg["cv"]["target_fpr"]
score_fn = lambda yt, ys: recall_at_fpr(yt, ys, max_fpr=target_fpr)

n_folds = cfg["cv"]["n_folds"]
freeze_epochs = cfg["model"].get("freeze_backbone_epochs", 0)

gkf = GroupKFold(n_splits=n_folds)
idx = np.arange(len(labels))
fold_scores = []

def freeze_backbone(model):
    for name, p in model.named_parameters():
        if "head" not in name and "classifier" not in name and "fc" not in name:
            p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

for fold, (tr, va) in enumerate(gkf.split(idx, labels, groups)):
    print(f"\n{'='*40} fold {fold} {'='*40}")

    train_ds = WindowDataset(
        [waveforms[i] for i in tr], labels[tr], groups[tr],
        feature_fn=feature_fn, cache_dir=cache_dir,
        transform=aug, sr=cfg["sr"],
    )
    val_ds = WindowDataset(
        [waveforms[i] for i in va], labels[va], groups[va],
        feature_fn=feature_fn, cache_dir=cache_dir,
        transform=None, sr=cfg["sr"],
    )

    m = cfg["model"]
    model = ASTClassifier(
        n_classes=2,
        pretrained=m.get("pretrained", True),
        target_length=m.get("target_length", 200),
        n_mels=cfg["feature"]["n_mels"],
    )

    # Class weighting for imbalance
    pos = float(labels[tr].mean())
    weights = np.array([1.0 / (1 - pos + 1e-6), 1.0 / (pos + 1e-6)], dtype=np.float32)

    # Phase 1: frozen backbone warmup (optional)
    if freeze_epochs > 0:
        print(f"  Frozen backbone for {freeze_epochs} epochs")
        freeze_backbone(model)
        frozen_cfg = TrainConfig(
            epochs=freeze_epochs,
            batch_size=cfg["train"]["batch_size"],
            lr=cfg["train"]["lr"] * 10,
            weight_decay=cfg["train"]["weight_decay"],
            patience=freeze_epochs,
            use_amp=cfg["train"]["use_amp"],
            scheduler="none",
            grad_clip=cfg["train"].get("grad_clip", 1.0),
            ckpt_path=str(ckpt_dir / f"fold{fold}_frozen.pt"),
        )
        fit(model, train_ds, val_ds, score_fn,
            cfg=frozen_cfg, device=device, class_weights=weights)
        unfreeze_all(model)

    # Phase 2: full fine-tuning
    remaining = cfg["train"]["epochs"] - freeze_epochs
    tcfg = TrainConfig(
        epochs=max(remaining, 1),
        batch_size=cfg["train"]["batch_size"],
        lr=cfg["train"]["lr"],
        weight_decay=cfg["train"]["weight_decay"],
        patience=cfg["train"]["patience"],
        use_amp=cfg["train"]["use_amp"],
        scheduler=cfg["train"]["scheduler"],
        grad_clip=cfg["train"].get("grad_clip", 1.0),
        ckpt_path=str(ckpt_dir / f"fold{fold}.pt"),
    )
    out = fit(model, train_ds, val_ds, score_fn,
              cfg=tcfg, device=device, class_weights=weights)

    print(f"  [fold {fold}] recall@{target_fpr:.0%}FPR = {out['best_score']:.4f}")
    fold_scores.append(out["best_score"])

## 7 — Results

In [ ]:
print(f"\n{'='*60}")
print(f"CV recall@{target_fpr:.0%}FPR: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")
print()
for i, s in enumerate(fold_scores):
    print(f"  fold {i}: {s:.4f}")

print(f"\nCheckpoints saved to: {ckpt_dir}")
print("Files:", list(ckpt_dir.glob("*.pt")))